# Tabela global de resultados de teste

Este notebook monta tabelas com os resultados de teste do modelo `Llama3.1-I`, separadas por metodo de recomendacao, juntando:
- o resultado `without_optimization`, exibido na primeira linha de cada metodo;
- e todos os resultados `with_optimization` encontrados para as configuracoes de `out/prompt_optimization/Llama3.1-I`.

A ideia aqui e ter uma visao consolidada por algoritmo, cobrindo `bprmf`, `item_knn`, `ncf` e `user_knn`.

In [1]:
import json
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists():
            return candidate
    raise FileNotFoundError(
        "Nao foi possivel localizar a raiz de explainability-with-LLMs. "
        "Execute o notebook a partir do projeto ou de um subdiretorio dele."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_NAME = "Llama3.1-I"
PROMPT_OPT_ROOT = PROJECT_ROOT / "out" / "prompt_optimization" / MODEL_NAME
TEST_ROOT = PROJECT_ROOT / "out" / "test_explainability"
WITHOUT_OPT_TEST_ROOT = TEST_ROOT / "without_optimization" / MODEL_NAME
WITH_OPT_TEST_ROOT = TEST_ROOT / "with_optimization" / MODEL_NAME

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PROMPT_OPT_ROOT: {PROMPT_OPT_ROOT}")
print(f"WITHOUT_OPT_TEST_ROOT: {WITHOUT_OPT_TEST_ROOT}")
print(f"WITH_OPT_TEST_ROOT: {WITH_OPT_TEST_ROOT}")

PROJECT_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs
PROMPT_OPT_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs/out/prompt_optimization/Llama3.1-I
WITHOUT_OPT_TEST_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs/out/test_explainability/without_optimization/Llama3.1-I
WITH_OPT_TEST_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs/out/test_explainability/with_optimization/Llama3.1-I


In [3]:
def lambda_to_float(lambda_name: str | None) -> float | None:
    if not lambda_name or not lambda_name.startswith("mmr_lambda_"):
        return None
    value = lambda_name.replace("mmr_lambda_", "")
    return float(value.replace("_", "."))


def find_named_parent(path: Path, prefix: str, default: str | None = None) -> str | None:
    for parent in path.parents:
        if parent.name.startswith(prefix):
            return parent.name
    return default


def discover_prompt_optimization_algorithms(prompt_opt_root: Path) -> list[str]:
    if not prompt_opt_root.exists():
        return []
    return sorted(path.name for path in prompt_opt_root.iterdir() if path.is_dir())


def load_test_metadata(metadata_path: Path) -> dict:
    return json.loads(metadata_path.read_text())


def discover_test_algorithms(test_root: Path) -> list[str]:
    if not test_root.exists():
        return []
    return sorted(path.name for path in test_root.iterdir() if path.is_dir())


def normalize_optional_value(value: object) -> object:
    if value is None or pd.isna(value):
        return pd.NA
    return value


def format_project_path(value: object) -> object:
    if value is None or pd.isna(value):
        return pd.NA

    path = Path(str(value))
    if not path.is_absolute():
        return str(path)

    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


PREFERRED_COLUMNS = [
    "optimization_mode",
    "algorithm",
    "llm_method",
    "metric",
    "metric_name",
    "metric_value",
    "repr_model",
    "early_profile",
    "mmr_lambda",
    "lambda_value",
    "mmr_pool",
    "prompt_source",
    "n_users",
    "time_to_explain",
    "best_prompt_path",
    "responses_metadata_path",
]


def discover_without_optimization_results(test_root: Path, valid_algorithms: list[str]) -> pd.DataFrame:
    rows = []

    for algorithm in valid_algorithms:
        metadata_path = test_root / algorithm / "sep" / "responses_metadata.json"
        if not metadata_path.exists():
            continue

        payload = load_test_metadata(metadata_path)
        args = payload.get("args", {})
        rows.append(
            {
                "optimization_mode": "without_optimization",
                "algorithm": algorithm,
                "metric": payload.get("metric", args.get("metric", "metric")),
                "metric_name": payload.get("metric_name", "METRIC"),
                "metric_value": payload.get("metric_value"),
                "repr_model": pd.NA,
                "early_profile": pd.NA,
                "mmr_lambda": pd.NA,
                "lambda_value": pd.NA,
                "mmr_pool": pd.NA,
                "prompt_source": payload.get("prompt_source", "desconhecido"),
                "llm_method": args.get("llm_method", "desconhecido"),
                "n_users": payload.get("n_users"),
                "time_to_explain": payload.get("time_to_explain"),
                "best_prompt_path": format_project_path(payload.get("best_prompt_path")),
                "responses_metadata_path": format_project_path(metadata_path),
            }
        )

    if not rows:
        return pd.DataFrame(columns=PREFERRED_COLUMNS)

    return pd.DataFrame.from_records(rows, columns=PREFERRED_COLUMNS)


def discover_with_optimization_results(test_root: Path, valid_algorithms: list[str]) -> pd.DataFrame:
    rows = []

    for algorithm in valid_algorithms:
        algorithm_dir = test_root / algorithm
        if not algorithm_dir.exists():
            continue

        for metadata_path in sorted(algorithm_dir.rglob("responses_metadata.json")):
            payload = load_test_metadata(metadata_path)
            args = payload.get("args", {})
            mmr_lambda = find_named_parent(metadata_path, "mmr_lambda_", None)

            rows.append(
                {
                    "optimization_mode": "with_optimization",
                    "algorithm": algorithm,
                    "metric": payload.get("metric", args.get("metric", "metric")),
                    "metric_name": payload.get("metric_name", "METRIC"),
                    "metric_value": payload.get("metric_value"),
                    "repr_model": normalize_optional_value(find_named_parent(metadata_path, "repr_", pd.NA)),
                    "early_profile": normalize_optional_value(find_named_parent(metadata_path, "early_", pd.NA)),
                    "mmr_lambda": normalize_optional_value(mmr_lambda),
                    "lambda_value": lambda_to_float(mmr_lambda),
                    "mmr_pool": normalize_optional_value(find_named_parent(metadata_path, "mmr_pool_", pd.NA)),
                    "prompt_source": payload.get("prompt_source", "desconhecido"),
                    "llm_method": args.get("llm_method", payload.get("best_prompt_model", "desconhecido")),
                    "n_users": payload.get("n_users"),
                    "time_to_explain": payload.get("time_to_explain"),
                    "best_prompt_path": format_project_path(payload.get("best_prompt_path")),
                    "responses_metadata_path": format_project_path(metadata_path),
                }
            )

    if not rows:
        return pd.DataFrame(columns=PREFERRED_COLUMNS)

    return pd.DataFrame.from_records(rows, columns=PREFERRED_COLUMNS)


def build_consolidated_table(prompt_opt_root: Path, without_opt_test_root: Path, with_opt_test_root: Path) -> pd.DataFrame:
    algorithms = sorted(
        set(discover_prompt_optimization_algorithms(prompt_opt_root))
        | set(discover_test_algorithms(without_opt_test_root))
        | set(discover_test_algorithms(with_opt_test_root))
    )
    without_opt = discover_without_optimization_results(without_opt_test_root, algorithms)
    with_opt = discover_with_optimization_results(with_opt_test_root, algorithms)

    rows = []
    if not without_opt.empty:
        rows.extend(without_opt.to_dict("records"))
    if not with_opt.empty:
        rows.extend(with_opt.to_dict("records"))

    if not rows:
        return pd.DataFrame(columns=PREFERRED_COLUMNS)

    consolidated = pd.DataFrame.from_records(rows, columns=PREFERRED_COLUMNS)

    text_columns = [
        "optimization_mode",
        "algorithm",
        "llm_method",
        "metric",
        "metric_name",
        "repr_model",
        "early_profile",
        "mmr_lambda",
        "mmr_pool",
        "prompt_source",
        "best_prompt_path",
        "responses_metadata_path",
    ]
    for column in text_columns:
        consolidated[column] = pd.Series(consolidated[column], dtype="string")

    consolidated["metric_value"] = pd.array(consolidated["metric_value"], dtype="Float64")
    consolidated["lambda_value"] = pd.array(consolidated["lambda_value"], dtype="Float64")
    consolidated["n_users"] = pd.array(consolidated["n_users"], dtype="Int64")
    consolidated["time_to_explain"] = pd.array(consolidated["time_to_explain"], dtype="Float64")

    consolidated["optimization_order"] = consolidated["optimization_mode"].map(
        {"without_optimization": 0, "with_optimization": 1}
    )
    consolidated = consolidated.sort_values(
        by=[
            "algorithm",
            "optimization_order",
            "early_profile",
            "repr_model",
            "lambda_value",
            "mmr_pool",
            "llm_method",
            "metric",
        ],
        na_position="last",
    ).reset_index(drop=True)

    return consolidated[PREFERRED_COLUMNS]


In [4]:
consolidated_table = build_consolidated_table(
    prompt_opt_root=PROMPT_OPT_ROOT,
    without_opt_test_root=WITHOUT_OPT_TEST_ROOT,
    with_opt_test_root=WITH_OPT_TEST_ROOT,
)

if consolidated_table.empty:
    warnings.warn("Nenhum resultado de teste foi encontrado para montar a tabela global.")
else:
    print(f"Linhas na tabela consolidada: {len(consolidated_table)}")
    for algorithm, algorithm_table in consolidated_table.groupby("algorithm", sort=False):
        print(f"\nAlgoritmo: {algorithm}")
        display(algorithm_table.reset_index(drop=True))

Linhas na tabela consolidada: 28

Algoritmo: bprmf


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,bprmf,Llama3.1-I,sep,SEP,0.642861,<NA>,<NA>,<NA>,<NA>,<NA>,default,122,222.167943,<NA>,out/test_explainability/without_optimization/L...
1,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.679508,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,239.303275,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,out/test_explainability/with_optimization/Llam...
2,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.683805,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,229.795681,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,out/test_explainability/with_optimization/Llam...
3,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.68766,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,225.473012,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,out/test_explainability/with_optimization/Llam...
4,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.668631,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,235.508255,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,out/test_explainability/with_optimization/Llam...
5,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.689605,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,239.414823,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,out/test_explainability/with_optimization/Llam...
6,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.689605,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,238.681194,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,out/test_explainability/with_optimization/Llam...



Algoritmo: item_knn


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,item_knn,Llama3.1-I,sep,SEP,0.582436,<NA>,<NA>,<NA>,<NA>,<NA>,default,122,221.63323,<NA>,out/test_explainability/without_optimization/L...
1,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.66817,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,234.104672,out/prompt_optimization/Llama3.1-I/item_knn/se...,out/test_explainability/with_optimization/Llam...
2,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.66817,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,235.73839,out/prompt_optimization/Llama3.1-I/item_knn/se...,out/test_explainability/with_optimization/Llam...
3,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.608547,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,221.98102,out/prompt_optimization/Llama3.1-I/item_knn/se...,out/test_explainability/with_optimization/Llam...
4,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.66817,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,235.719262,out/prompt_optimization/Llama3.1-I/item_knn/se...,out/test_explainability/with_optimization/Llam...
5,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.665992,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,261.31124,out/prompt_optimization/Llama3.1-I/item_knn/se...,out/test_explainability/with_optimization/Llam...
6,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.608547,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,220.57902,out/prompt_optimization/Llama3.1-I/item_knn/se...,out/test_explainability/with_optimization/Llam...



Algoritmo: ncf


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,ncf,Llama3.1-I,sep,SEP,0.596754,<NA>,<NA>,<NA>,<NA>,<NA>,default,122,220.366039,<NA>,out/test_explainability/without_optimization/L...
1,with_optimization,ncf,Llama3.1-I,sep,SEP,0.639159,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,227.978361,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,out/test_explainability/with_optimization/Llam...
2,with_optimization,ncf,Llama3.1-I,sep,SEP,0.636248,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,233.159266,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,out/test_explainability/with_optimization/Llam...
3,with_optimization,ncf,Llama3.1-I,sep,SEP,0.621337,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,228.570212,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,out/test_explainability/with_optimization/Llam...
4,with_optimization,ncf,Llama3.1-I,sep,SEP,0.651415,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,227.713135,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,out/test_explainability/with_optimization/Llam...
5,with_optimization,ncf,Llama3.1-I,sep,SEP,0.639307,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,225.448595,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,out/test_explainability/with_optimization/Llam...
6,with_optimization,ncf,Llama3.1-I,sep,SEP,0.644946,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,225.471862,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,out/test_explainability/with_optimization/Llam...



Algoritmo: user_knn


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,user_knn,Llama3.1-I,sep,SEP,0.625719,<NA>,<NA>,<NA>,<NA>,<NA>,default,122,223.491391,<NA>,out/test_explainability/without_optimization/L...
1,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.659913,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,225.039552,out/prompt_optimization/Llama3.1-I/user_knn/se...,out/test_explainability/with_optimization/Llam...
2,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.68759,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,237.164812,out/prompt_optimization/Llama3.1-I/user_knn/se...,out/test_explainability/with_optimization/Llam...
3,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.672071,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,235.875741,out/prompt_optimization/Llama3.1-I/user_knn/se...,out/test_explainability/with_optimization/Llam...
4,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.68759,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,236.700224,out/prompt_optimization/Llama3.1-I/user_knn/se...,out/test_explainability/with_optimization/Llam...
5,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.673398,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,234.51544,out/prompt_optimization/Llama3.1-I/user_knn/se...,out/test_explainability/with_optimization/Llam...
6,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.677381,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,242.298632,out/prompt_optimization/Llama3.1-I/user_knn/se...,out/test_explainability/with_optimization/Llam...


In [5]:
def best_metric_for_mode(table: pd.DataFrame, mode: str) -> float | object:
    values = pd.to_numeric(
        table.loc[table["optimization_mode"] == mode, "metric_value"],
        errors="coerce",
    ).dropna()
    if values.empty:
        return pd.NA
    return float(values.max())


preferred_algorithm_order = ["user_knn", "item_knn", "bprmf", "ncf"]
available_algorithms = consolidated_table["algorithm"].dropna().astype(str).unique().tolist()
algorithm_order = [
    algorithm
    for algorithm in preferred_algorithm_order
    if algorithm in available_algorithms
]
algorithm_order.extend(
    algorithm for algorithm in available_algorithms if algorithm not in algorithm_order
)

summary_values = {}
for algorithm in algorithm_order:
    algorithm_table = consolidated_table.loc[consolidated_table["algorithm"] == algorithm]
    summary_values[algorithm] = [
        best_metric_for_mode(algorithm_table, "without_optimization"),
        best_metric_for_mode(algorithm_table, "with_optimization"),
    ]

best_before_after_table = pd.DataFrame(
    summary_values,
    index=["initial_system_prompt", "best_system_prompt"],
).astype("Float64").round(6)
best_before_after_table.index.name = None
best_before_after_table.columns = pd.MultiIndex.from_product(
    [["test_metric_value"], best_before_after_table.columns]
)

display(best_before_after_table)


test_metric_value                              
                               user_knn  item_knn     bprmf       ncf
initial_system_prompt          0.625719  0.582436  0.642861  0.596754
best_system_prompt              0.68759   0.66817  0.689605  0.651415